# Connectivity Notebook — Storage Stress (Natural Gas Calendar Spreads)

Working API calls for every data source and the backtest platform.

Showing connectivity for..
1. EIA v2 natural-gas endpoints (weekly salt storage + Henry Hub futures) — exact routes, params, and response parsing.
2. IBKR connectivity via **`ib_async`** (the maintained successor to `ib_insync`, author has passed).
3. A seeded **synthetic fallback** so the whole project runs with *no* key and *no* gateway, then swaps to live by setting one env var.

> **Platform decision:** IBKR is the default and it *does* support the strategy — NYMEX `NG` futures and futures calendar spreads (combo orders) are tradable through TWS/IB Gateway, and paper trading is free. So we keep IBKR. `ib_async` replaces the proposal's `ib_insync` (unmaintained since its author's death in early 2024).

In [ ]:
import os, sys
sys.path.append("../src")
import importlib, connectivity as C
importlib.reload(C)
from dotenv load_dotenv; load_dotenv()
print("EIA_API_KEY set:", bool(os.environ.get("EIA_API_KEY")))
print("-> with no key, every call below returns SEEDED SYNTHETIC data.")
print("-> get a free key at https://www.eia.gov/opendata/ and `export EIA_API_KEY=...`")

## 1. EIA — weekly salt-region storage (the core feed)

* Route: `GET https://api.eia.gov/v2/natural-gas/stor/wkly/data/`
* Series facet: `NW2_EPG0_SSO_R33_BCF_W` (Salt, South Central, working gas, Bcf)
* Auth: `api_key` query param (free)
* Cadence: weekly, Thursday 10:30 ET release

In [ ]:
salt = C.eia_storage("salt_south_central", start="2017-01-01")
print("rows:", len(salt), "| last date:", salt.index[-1].date())
salt.tail()

**Live request shape** (what runs when a key is present):

```
GET https://api.eia.gov/v2/natural-gas/stor/wkly/data/
    ?api_key=YOUR_KEY
    &frequency=weekly
    &data[0]=value
    &facets[series][]=NW2_EPG0_SSO_R33_BCF_W
    &start=2017-01-01
    &sort[0][column]=period&sort[0][direction]=asc
    &length=5000
```
Response JSON path: `response.data[*].{period, value}` → level (Bcf); we diff to get weekly net flow.

## 2. EIA — Henry Hub futures (spread legs + sanity)

* Route: `natural-gas/pri/fut`, series `RNGC1` (front), `RNGC2` (2nd) — daily settle $/MMBtu.
* We use these to build the **C1−C2 calendar spread** the strategy trades.

In [ ]:
c1 = C.eia_henryhub_futures(1, start="2017-01-01")
c2 = C.eia_henryhub_futures(2, start="2017-01-01")
spread = (c1.iloc[:,0] - c2.iloc[:,0]).rename("cal_spread_c1_c2")
print("front:", round(c1.iloc[-1,0],3), " 2nd:", round(c2.iloc[-1,0],3),
      " spread:", round(spread.iloc[-1],3), "$/MMBtu")
spread.tail()

## 3. Supporting feeds (degree days, regional basis)

NOAA degree days drive the demand-regime weight; Waha / Dom South basis feed the
residualization (section 5). Both have synthetic seasonal/mean-reverting models so
the orthogonality machinery is testable today; swap to live (NOAA CDO / FRED, ICE
EOD / EIA bidweek) when access is in place.

In [ ]:
dd = C.noaa_degree_days(synthetic=True)
waha = C.regional_basis("waha"); dom = C.regional_basis("domsouth")
print("degree days rows:", len(dd), "| last HDD/CDD:",
      int(dd['hdd'].iloc[-1]), int(dd['cdd'].iloc[-1]))
print("waha basis last:", round(waha.iloc[-1,0],3),
      "| dom south last:", round(dom.iloc[-1,0],3))

## 4. IBKR connectivity (trading / paper) via `ib_async`

Steps to make the **live** call work:
1. `pip install ib_async`
2. Install **IB Gateway** (or TWS), log into your **paper** account.
3. Configure → API → *Enable ActiveX and Socket Clients*; trust `127.0.0.1`; note the socket port (IB Gateway paper = **4002**, TWS paper = **7497**).
4. Run the cell below.

The function qualifies the front two `NG` NYMEX futures and pulls 30 daily bars on the front contract.

In [ ]:
try:
    import ib_async
    print("ib_async", ib_async.__version__, "importable.")
    print("Start IB Gateway (paper) then run:")
    print(">>> C.ibkr_smoke_test(C.IBKRConfig(port=4002))")
    # Uncomment when a paper gateway is running:
    # out = C.ibkr_smoke_test(C.IBKRConfig(port=4002))
    # print(out['connected'], out['front'], out['second']); out['bars'].tail()
except ImportError:
    print("ib_async not installed in this kernel.")
    print("Install with: pip install ib_async   (NOT ib_insync — unmaintained)")

### Why IBKR clears the strategy's requirements
| Requirement | IBKR support |
|---|---|
| NYMEX `NG` futures | Yes — full futures market access |
| **Calendar spreads** | Yes — native futures combo (BAG) orders, or two legs |
| Paper trading | Yes — free paper account mirrors live |
| Programmatic API | Yes — TWS API via `ib_async` |
| Historical bars for backtest seed | Yes — `reqHistoricalData` (limited history; pair with EIA/CME settles) |

